
# Holistic Data Preparer

**Dataset Files Used**
- customer_credit_risk.csv
- customer_metadata.json
- loan_history.db


# Part A: Theory Questions

# Q1: Explain the given topics


## 1. What is Data Analysis?

Data Analysis is the process of collecting, cleaning, transforming and interpreting data
to extract meaningful insights and support decision making.

### Steps:
1. Data Collection
2. Data Cleaning
3. Data Transformation
4. Analysis
5. Visualization
6. Decision Making



## 2. Planning a Data Science Project

A Data Science project should follow:

1. Problem Definition
2. Data Collection
3. Data Cleaning
4. Feature Engineering
5. Model Building
6. Evaluation
7. Deployment



## 3. Framing a Machine Learning Problem

Machine learning problems are framed as:

- Classification
- Regression
- Clustering
- Recommendation

Example:

Predicting loan default is a **classification** problem.


# Q2: Explain Tensors with NumPy Examples

## What is a Tensor?

A tensor is a mathematical structure used to represent data in multiple dimensions.

It is a generalization of:

- Scalar → 0D tensor
- Vector → 1D tensor
- Matrix → 2D tensor
- Higher dimensional arrays → 3D+ tensors

Tensors are widely used in machine learning and deep learning because they efficiently store and process numerical data.

Examples:
- Scalar: single value
- Vector: list of values
- Matrix: table of values
- 3D Tensor: stack of matrices

Important tensor properties:
- Dimension
- Shape
- Data type

## Tensor Example using NumPy

In [1]:
import numpy as np

# Scalar (0D)
scalar = np.array(10)
print("Scalar:", scalar)
print("Dimension:", scalar.ndim)
print("Shape:", scalar.shape)
print()

# Vector (1D)
vector = np.array([1, 2, 3, 4])
print("Vector:", vector)
print("Dimension:", vector.ndim)
print("Shape:", vector.shape)
print()

# Matrix (2D)
matrix = np.array([[1, 2], [3, 4]])
print("Matrix:\n", matrix)
print("Dimension:", matrix.ndim)
print("Shape:", matrix.shape)
print()

# 3D Tensor
tensor_3d = np.array([
    [[1, 2], [3, 4]],
    [[5, 6], [7, 8]]
])

print("3D Tensor:\n", tensor_3d)
print("Dimension:", tensor_3d.ndim)
print("Shape:", tensor_3d.shape)

Scalar: 10
Dimension: 0
Shape: ()

Vector: [1 2 3 4]
Dimension: 1
Shape: (4,)

Matrix:
 [[1 2]
 [3 4]]
Dimension: 2
Shape: (2, 2)

3D Tensor:
 [[[1 2]
  [3 4]]

 [[5 6]
  [7 8]]]
Dimension: 3
Shape: (2, 2, 2)


# Part B: Data Acquisition

In [2]:
import pandas as pd
import sqlite3
import json
import requests
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, MissingIndicator
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler, MaxAbsScaler, RobustScaler, Normalizer, Binarizer, PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
from scipy.stats import zscore, boxcox
from scipy.stats.mstats import winsorize

## Load CSV File

In [3]:
df = pd.read_csv("./Dataset/customer_credit_risk.csv")
df.head()

,customer_id,age,gender,region,education_level,employment_type,annual_income,loan_amount,loan_purpose,credit_score,repayment_history,transaction_count,spending_ratio,join_date,default_flag
0,C1000,59.0,Other,South,Secondary,Unemployed,744030.94,317948.42,Other,664.522475,2,150,80.45,2019-04-02,0
1,C1001,49.0,Male,South,Secondary,Self-Employed,2493237.84,550580.94,Car,845.459027,5,35,33.45,2018-04-13,1
2,C1002,35.0,Female,South,Secondary,NaN,488693.29,127668.66,Education,577.095463,3,123,80.98,2021-01-31,0
3,C1003,28.0,Male,East,Graduate,NaN,NaN,358062.00,Education,680.074516,1,98,90.53,2020-09-30,1
4,C1004,41.0,Male,West,Secondary,Salaried,343915.13,249095.35,Education,588.513210,7,34,45.18,2020-07-03,0


## Load JSON File

In [4]:
with open("./Dataset/customer_metadata.json") as f:
    metadata = json.load(f)
meta_df = pd.DataFrame(metadata)
meta_df.head()

,customer_id,marital_status,dependents,preferred_contact
0,C1000,Married,1,Phone
1,C1001,Single,3,Email
2,C1002,Single,0,Phone
3,C1003,Married,2,SMS
4,C1004,Married,0,Phone


## Load SQL Database

In [5]:
conn = sqlite3.connect("./Dataset/loan_history.db")
sql_df = pd.read_sql("SELECT * FROM loan_payments", conn)
sql_df.head()

,payment_id,customer_id,payment_status,payment_amount
0,1,C1456,Paid,49836.43
1,2,C1292,Defaulted,46241.55
2,3,C1502,Paid,10518.68
3,4,C1873,Paid,25341.92
4,5,C1920,Defaulted,25030.00


## Load API Data

In [6]:
url = "https://api.worldbank.org/v2/country/IND/indicator/FP.CPI.TOTL.ZG?format=json"
api_data = requests.get(url).json()
print(api_data[:1])

[{'page': 1, 'pages': 2, 'per_page': 50, 'total': 66, 'sourceid': '2', 'lastupdated': '2026-07-13'}]


# Part C: Data Understanding and Cleaning

In [7]:
print(df.info())
print(df.describe())
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        1000 non-null   object 
 1   age                920 non-null    float64
 2   gender             950 non-null    object 
 3   region             1000 non-null   object 
 4   education_level    1000 non-null   object 
 5   employment_type    940 non-null    object 
 6   annual_income      900 non-null    float64
 7   loan_amount        1000 non-null   float64
 8   loan_purpose       1000 non-null   object 
 9   credit_score       930 non-null    float64
 10  repayment_history  1000 non-null   int64  
 11  transaction_count  1000 non-null   int64  
 12  spending_ratio     1000 non-null   float64
 13  join_date          1000 non-null   object 
 14  default_flag       1000 non-null   int64  
dtypes: float64(5), int64(3), object(7)
memory usage: 117.3+ KB
None
         

## Generate Data Quality Report

In [8]:
from ydata_profiling import ProfileReport
profile = ProfileReport(df, title="Data Quality Report", explorative=True)
profile.to_file("data_quality_report.html")

C:\Users\admin\AppData\Local\Temp\ipykernel_22384\4192873407.py:1: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 15/15 [00:00<00:00, 738.95it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

## Complete Case Analysis

In [ ]:
print("Before:", df.shape)
df_complete = df.copy().dropna()
print("After:", df_complete.shape)

Before: (1000, 15)
After: (688, 15)


## Mean Imputation

In [ ]:
df_simple = df.copy()
num_cols = ['age','annual_income','credit_score']
print("Before")
print(df_simple[num_cols].isnull().sum())

imp = SimpleImputer(strategy="mean")
df_simple[num_cols] = imp.fit_transform(df_simple[num_cols])

print("\nAfter")
print(df_simple[num_cols].isnull().sum())

Before
age               80
annual_income    100
credit_score      70
dtype: int64

After
age              0
annual_income    0
credit_score     0
dtype: int64


## Most Frequent Imputation

In [ ]:
df_freq = df.copy()
cat_cols = ['gender','employment_type']
print("Before")
print(df_freq[cat_cols].isnull().sum())

imp = SimpleImputer(strategy="most_frequent")
df_freq[cat_cols] = imp.fit_transform(df_freq[cat_cols])

print("\nAfter")
print(df_freq[cat_cols].isnull().sum())

Before
gender             50
employment_type    60
dtype: int64

After
gender             0
employment_type    0
dtype: int64


## Random Sample Imputation

In [ ]:
df_random = df.copy()
for col in ['age','annual_income','credit_score']:
    df_random[col] = df_random[col].apply(lambda x: df_random[col].dropna().sample(1).iloc[0] if pd.isnull(x) else x)
print(df_random[['age','annual_income','credit_score']].isnull().sum())

age              0
annual_income    0
credit_score     0
dtype: int64


## KNN Imputation

In [ ]:
df_knn = df.copy()
knn = KNNImputer()
df_knn[['annual_income','loan_amount','credit_score']] = knn.fit_transform(df_knn[['annual_income','loan_amount','credit_score']])
print(df_knn[['annual_income','loan_amount','credit_score']].isnull().sum())

annual_income    0
loan_amount      0
credit_score     0
dtype: int64


## MICE Imputation

In [ ]:
df_mice = df.copy()
mice = IterativeImputer(random_state=42)
df_mice[['annual_income','loan_amount','credit_score']] = mice.fit_transform(df_mice[['annual_income','loan_amount','credit_score']])
print(df_mice[['annual_income','loan_amount','credit_score']].isnull().sum())

annual_income    0
loan_amount      0
credit_score     0
dtype: int64


# Part D: Outlier Handling

## Z-Score Method

In [ ]:
df_z = df.copy()
df_z['zscore'] = zscore(df_z['annual_income'].fillna(df_z['annual_income'].mean()))
outliers = df_z[df_z['zscore'].abs() > 3]
print(outliers.shape)

(8, 16)


## IQR Method

In [ ]:
df_iqr = df.copy()
Q1 = df_iqr['annual_income'].quantile(0.25)
Q3 = df_iqr['annual_income'].quantile(0.75)
IQR = Q3 - Q1
outliers = df_iqr[(df_iqr['annual_income'] < Q1 - 1.5*IQR) | (df_iqr['annual_income'] > Q3 + 1.5*IQR)]
print(outliers.shape)

(41, 15)


## Percentile Method

In [ ]:
df_percentile = df.copy()
lower = df_percentile['annual_income'].quantile(0.01)
upper = df_percentile['annual_income'].quantile(0.99)
df_percentile['annual_income'] = df_percentile['annual_income'].clip(lower, upper)
print(df_percentile['annual_income'].describe())

count    9.000000e+02
mean     6.007291e+05
std      3.788220e+05
min      1.545848e+05
25%      3.469426e+05
50%      5.104346e+05
75%      7.405156e+05
max      2.254826e+06
Name: annual_income, dtype: float64


## Winsorization

In [ ]:
df_winsor = df.copy()
df_winsor['annual_income'] = winsorize(df_winsor['annual_income'].fillna(df_winsor['annual_income'].mean()), limits=[0.05,0.05])
print(df_winsor['annual_income'].describe())

count    1.000000e+03
mean     5.789290e+05
std      2.703980e+05
min      2.094580e+05
25%      3.606969e+05
50%      5.548340e+05
75%      7.046050e+05
max      1.203457e+06
Name: annual_income, dtype: float64


C:\Users\ASUS\AppData\Roaming\Python\Python311\site-packages\numpy\lib\function_base.py:4824: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(


# Part E: Encoding and Feature Engineering

In [ ]:
df_encode = df.copy()
le = LabelEncoder()
df_encode['gender_encoded'] = le.fit_transform(df_encode['gender'].fillna('Unknown'))
print(df_encode[['gender','gender_encoded']].head())

   gender  gender_encoded
0   Other               2
1    Male               1
2  Female               0
3    Male               1
4    Male               1


In [ ]:
ord_enc = OrdinalEncoder(categories=[['Secondary','Graduate','Post-Graduate']])
df_encode['education_encoded'] = ord_enc.fit_transform(df_encode[['education_level']])
print(df_encode[['education_level','education_encoded']].head())

  education_level  education_encoded
0       Secondary                0.0
1       Secondary                0.0
2       Secondary                0.0
3        Graduate                1.0
4       Secondary                0.0


In [ ]:
df_encode = pd.get_dummies(df_encode, columns=['region','loan_purpose'], drop_first=True)
print(df_encode.head())

  customer_id   age  gender education_level employment_type  annual_income  \
0       C1000  59.0   Other       Secondary      Unemployed      744030.94   
1       C1001  49.0    Male       Secondary   Self-Employed     2493237.84   
2       C1002  35.0  Female       Secondary             NaN      488693.29   
3       C1003  28.0    Male        Graduate             NaN            NaN   
4       C1004  41.0    Male       Secondary        Salaried      343915.13   

   loan_amount  credit_score  repayment_history  transaction_count  ...  \
0    317948.42    664.522475                  2                150  ...   
1    550580.94    845.459027                  5                 35  ...   
2    127668.66    577.095463                  3                123  ...   
3    358062.00    680.074516                  1                 98  ...   
4    249095.35    588.513210                  7                 34  ...   

   default_flag gender_encoded  education_encoded  region_North  region_South  \

In [ ]:
df_encode['join_date'] = pd.to_datetime(df_encode['join_date'])
df_encode['year'] = df_encode['join_date'].dt.year
df_encode['month'] = df_encode['join_date'].dt.month
print(df_encode[['join_date','year','month']].head())

   join_date  year  month
0 2019-04-02  2019      4
1 2018-04-13  2018      4
2 2021-01-31  2021      1
3 2020-09-30  2020      9
4 2020-07-03  2020      7


In [ ]:
df_encode['income_bin'] = pd.cut(df_encode['annual_income'], bins=5, labels=False)
df_encode['quantile_bin'] = pd.qcut(df_encode['annual_income'], q=4, labels=False, duplicates='drop')
print(df_encode[['annual_income','income_bin','quantile_bin']].head())

   annual_income  income_bin  quantile_bin
0      744030.94         0.0           3.0
1     2493237.84         1.0           3.0
2      488693.29         0.0           1.0
3            NaN         NaN           NaN
4      343915.13         0.0           0.0


In [ ]:
binr = Binarizer(threshold=700)
df_encode['credit_binary'] = binr.fit_transform(df_encode[['credit_score']].fillna(700))
print(df_encode[['credit_score','credit_binary']].head())

   credit_score  credit_binary
0    664.522475            0.0
1    845.459027            1.0
2    577.095463            0.0
3    680.074516            0.0
4    588.513210            0.0


In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42)
df_encode['transaction_cluster'] = kmeans.fit_predict(df_encode[['transaction_count']])
print(df_encode[['transaction_count','transaction_cluster']].head())

   transaction_count  transaction_cluster
0                150                    2
1                 35                    1
2                123                    2
3                 98                    0
4                 34                    1


# Part F: Scaling

In [ ]:
df_scale = df.copy()
scaler = StandardScaler()
df_scale['std_income'] = scaler.fit_transform(df_scale[['annual_income']].fillna(df_scale[['annual_income']].mean()))
print(df_scale[['annual_income','std_income']].head())

   annual_income  std_income
0      744030.94    0.161848
1     2493237.84    2.842779
2      488693.29   -0.229497
3            NaN    0.000000
4      343915.13   -0.451392


In [ ]:
minmax = MinMaxScaler()
df_scale['minmax_income'] = minmax.fit_transform(df_scale[['annual_income']].fillna(df_scale[['annual_income']].mean()))
print(df_scale[['annual_income','minmax_income']].head())

   annual_income  minmax_income
0      744030.94       0.069821
1     2493237.84       0.255801
2      488693.29       0.042673
3            NaN       0.058593
4      343915.13       0.027280


In [ ]:
maxabs = MaxAbsScaler()
df_scale['maxabs_income'] = maxabs.fit_transform(df_scale[['annual_income']].fillna(df_scale[['annual_income']].mean()))
print(df_scale[['annual_income','maxabs_income']].head())

   annual_income  maxabs_income
0      744030.94       0.078379
1     2493237.84       0.262648
2      488693.29       0.051481
3            NaN       0.067255
4      343915.13       0.036229


In [ ]:
robust = RobustScaler()
df_scale['robust_income'] = robust.fit_transform(df_scale[['annual_income']].fillna(df_scale[['annual_income']].mean()))
print(df_scale[['annual_income','robust_income']].head())

   annual_income  robust_income
0      744030.94       0.550138
1     2493237.84       5.636400
2      488693.29      -0.192321
3            NaN       0.243081
4      343915.13      -0.613300


# Part G: Transformations and Feature Construction

In [ ]:
df_trans = df.copy()
df_trans['log_income'] = np.log1p(df_trans['annual_income'])
df_trans['sqrt_loan'] = np.sqrt(df_trans['loan_amount'])
df_trans['reciprocal_income'] = 1 / df_trans['annual_income']
print(df_trans[['annual_income','log_income']].head())

   annual_income  log_income
0      744030.94   13.519839
1     2493237.84   14.729093
2      488693.29   13.099492
3            NaN         NaN
4      343915.13   12.748153


In [ ]:
positive_income = df_trans['annual_income'].fillna(df_trans['annual_income'].mean()) + 1
df_trans['boxcox_income'], lam = boxcox(positive_income)
print(df_trans[['annual_income','boxcox_income']].head())

   annual_income  boxcox_income
0      744030.94       5.051338
1     2493237.84       5.145618
2      488693.29       5.013431
3            NaN       5.037867
4      343915.13       4.979460


In [ ]:
pt = PowerTransformer(method='yeo-johnson')
df_trans['yeojohnson_income'] = pt.fit_transform(df_trans[['annual_income']].fillna(df_trans[['annual_income']].mean()))
print(df_trans[['annual_income','yeojohnson_income']].head())

   annual_income  yeojohnson_income
0      744030.94           0.649697
1     2493237.84           2.462322
2      488693.29          -0.079108
3            NaN           0.390694
4      343915.13          -0.732227


In [ ]:
df_trans['debt_to_income'] = df_trans['loan_amount'] / df_trans['annual_income']
print(df_trans[['loan_amount','annual_income','debt_to_income']].head())

   loan_amount  annual_income  debt_to_income
0    317948.42      744030.94        0.427332
1    550580.94     2493237.84        0.220830
2    127668.66      488693.29        0.261245
3    358062.00            NaN             NaN
4    249095.35      343915.13        0.724293


# Part H: Final Export

## cleaned dataset

In [ ]:
# Part H: Final Export

final_df = df.copy()

num_cols = ['age', 'annual_income', 'credit_score', 'loan_amount']
cat_cols = ['gender', 'employment_type']

# Basic cleaning
final_df[num_cols] = SimpleImputer(strategy='mean').fit_transform(
    final_df[num_cols]
)

final_df[cat_cols] = SimpleImputer(strategy='most_frequent').fit_transform(
    final_df[cat_cols]
)

final_df['annual_income'] = winsorize(
    final_df['annual_income'],
    limits=[0.05, 0.05]
)

final_df['gender_encoded'] = LabelEncoder().fit_transform(
    final_df['gender']
)

# Export cleaned dataset
final_df.to_csv("final_cleaned_dataset.csv", index=False)

# Create scaled dataset
scaled_df = final_df.copy()
scaled_df[num_cols] = StandardScaler().fit_transform(
    scaled_df[num_cols]
)

scaled_df.to_csv("scaled_dataset.csv", index=False)

print("Files exported successfully")

Files exported successfully


# Project Summary and Conclusion

## Holistic Data Preparer

This project implemented a complete data preprocessing pipeline for preparing raw data for machine learning and analytical tasks.

### Key Tasks Performed
- Data acquisition from CSV, JSON, SQL database, and API
- Data quality analysis and profiling
- Missing value handling
- Outlier detection and treatment
- Encoding categorical variables
- Numerical binning and transformations
- Feature scaling and normalization
- Feature engineering
- Final dataset export

---

## Best Methods Identified

After comparing multiple preprocessing techniques, the following methods were found to be the most suitable for this dataset:

### Missing Value Handling
- **Mean Imputation** performed well for numerical columns because missing values were moderate and data distribution was fairly stable.
- **Most Frequent Imputation** was suitable for categorical columns such as gender and employment type.

### Outlier Handling
- **Winsorization** was the most effective because it reduced the effect of extreme annual income values without removing rows.

### Encoding
- **One Hot Encoding** was best for nominal categorical variables such as region and loan purpose.
- **Ordinal Encoding** was suitable for ordered variables such as education level.

### Scaling
- **StandardScaler** was the most appropriate because numerical features had different ranges and needed normalization for machine learning readiness.

### Transformations
- **Yeo-Johnson** and **Box-Cox** improved skewed numerical distributions effectively.

---

## Final Outputs Generated
- `final_cleaned_dataset.csv`
- `scaled_dataset.csv`
- `data_quality_report.html`

---

## Conclusion
The dataset was successfully transformed from raw and inconsistent data into a clean, structured, and machine-learning-ready format.

This project demonstrates a holistic approach to data preparation by evaluating multiple preprocessing techniques and selecting the most effective methods for the given dataset.